In [1]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [2]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по яйцам v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Яйца
379,АЛМАТИНСКАЯ ОБЛАСТЬ,2025-06-01,38080.1
2020,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2018-09-01,71148.5
487,АТЫРАУСКАЯ ОБЛАСТЬ,2023-11-01,453.2
807,ГАСТАНА,2018-10-01,4.4
150,АКТЮБИНСКАЯ ОБЛАСТЬ,2016-12-01,14131.7
886,ГАСТАНА,2025-07-01,0.2
1278,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2019-04-01,61770.9
850,ГАСТАНА,2022-06-01,0.4
1354,КОСТАНАЙСКАЯ ОБЛАСТЬ,2015-01-01,37858.0
1240,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2016-02-01,51383.0


In [3]:
# === загружаем данные ===
best_methods = pd.read_excel("results/Яйца - Лучшие модели (MAPE_then_MAE) v2.xlsx")  # лучшие методы
best_methods

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АКМОЛИНСКАЯ ОБЛАСТЬ,4.42,2369.14,4.70,2489.20,4.44,2520.45,HW,MAPE,4.42,2369.14
1,АКТЮБИНСКАЯ ОБЛАСТЬ,6.64,1272.95,7.65,1465.02,12.02,2292.86,HW,MAPE,6.64,1272.95
2,АТЫРАУСКАЯ ОБЛАСТЬ,12.95,503.07,19.69,752.81,80.04,3188.79,HW,MAPE,12.95,503.07
3,ГШЫМКЕНТ,9.08,1695.09,13.82,2546.49,9.49,1798.74,HW,MAPE,9.08,1695.09
4,ЖАМБЫЛСКАЯ ОБЛАСТЬ,13.57,1059.44,20.33,1383.31,24.05,1487.32,HW,MAPE,13.57,1059.44
5,КАРАГАНДИНСКАЯ ОБЛАСТЬ,6.58,3350.80,6.68,3347.73,8.83,4483.22,HW,MAPE,6.58,3347.73
6,ОБЛАСТЬ АБАЙ,8.12,376.58,11.98,537.52,21.98,905.41,HW,MAPE,8.12,376.58
7,ОБЛАСТЬ ЖЕТІСУ,5.67,1582.11,8.58,2396.53,19.13,5290.43,HW,MAPE,5.67,1582.11
8,ОБЛАСТЬ ҰЛЫТАУ,29.40,542.70,37.34,785.80,49.84,878.03,HW,MAPE,29.40,542.70
9,ПАВЛОДАРСКАЯ ОБЛАСТЬ,4.85,777.12,5.28,806.74,6.07,955.97,HW,MAPE,4.85,777.12


In [4]:
actual_aug = pd.read_excel("Яйца 08.2025.xlsx")
actual_aug["Период"] = pd.to_datetime(actual_aug["Период"], format="%Y-%m")
actual_aug["Яйца"] = (actual_aug["Яйца"]
                     .astype(str)
                     .str.replace(".", "", regex=False)   # убираем разделители тысяч
                     .str.replace(",", ".", regex=False)  # заменяем запятую на точку
                     .astype(float))
actual_aug.to_excel("Яйца обработанные август 2025.xlsx", index=False)
actual_aug



,Регион,Период,Яйца
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2025-08-01,58844.8
1,АКТЮБИНСКАЯ ОБЛАСТЬ,2025-08-01,20835.1
2,АЛМАТИНСКАЯ ОБЛАСТЬ,2025-08-01,42994.8
3,АТЫРАУСКАЯ ОБЛАСТЬ,2025-08-01,4209.1
4,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2025-08-01,12202.0
5,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2025-08-01,14890.0
6,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2025-08-01,54429.9
7,КОСТАНАЙСКАЯ ОБЛАСТЬ,2025-08-01,32694.5
8,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2025-08-01,850.8
9,МАНГИСТАУСКАЯ ОБЛАСТЬ,2025-08-01,58.2


In [5]:
# === настройки ===
TARGET = "Яйца"
CUTOFF = "2025-07-01"
FORECAST = "2025-08-01"
EPS = 1e-6
SEAS = 12

In [6]:
# оставляем только август 2025 для проверки
fact_aug = (actual_aug[actual_aug["Период"] == "2025-08-01"]
            .set_index("Регион")[TARGET])


In [7]:
# === функции прогнозов, строго как в обучающем коде ===
def fc_hw_like_training(train):
    # train — Series с MS частотой
    train_log = np.log1p(train)  # log1p
    model = ExponentialSmoothing(train_log, seasonal="add", seasonal_periods=SEAS)\
            .fit(optimized=True)
    fc_log = model.forecast(1)
    return float(np.expm1(fc_log).iloc[0])  # expm1

In [8]:
def fc_sarima_like_training(train):
    train_plus = train + EPS
    train_log  = np.log(train_plus)
    use_seasonal = len(train_log) >= 2 * SEAS

    sar = auto_arima(
        train_log,
        seasonal=use_seasonal,
        m=SEAS if use_seasonal else 1,
        D=1 if use_seasonal else 0,
        seasonal_test=None,
        boxcox=True,            # как в обучении
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore"
    )

    fc_log = sar.predict(n_periods=1)
    # берём первый элемент позиционно, независимо от типа (Series/ndarray/scalar)
    fc_log_scalar = np.asarray(fc_log).ravel()[0]

    return float(np.exp(fc_log_scalar) - EPS)

In [9]:
def fc_prophet_like_training(train):
    df_p = (train.reset_index()
                 .rename(columns={"Период": "ds", TARGET: "y"}))
    df_p["y"] = np.log(df_p["y"] + EPS)       # лог как в обучении
    m = Prophet()
    m.fit(df_p)
    future = m.make_future_dataframe(periods=1, freq="MS")
    yhat_log = m.predict(future)["yhat"].iloc[-1]
    return float(np.exp(yhat_log) - EPS)

In [10]:
methods_map = {
    "HW": fc_hw_like_training,
    "Holt-Winters": fc_hw_like_training,
    "Holt_Winters": fc_hw_like_training,
    "SARIMA": fc_sarima_like_training,
    "Prophet": fc_prophet_like_training,
}

In [11]:
# === прогон по регионам согласно «лучшему методу» ===
rows = []
for _, r in best_methods.iterrows():
    region = r["Регион"]
    method = r["Best_method"]

    ts = (df[df["Регион"] == region]
          .set_index("Период")[TARGET]
          .asfreq("MS")
          .sort_index())

    train = ts[:CUTOFF].dropna()
    if len(train) < 24:
        # как и в обучении, пропускаем короткие ряды
        continue

    # вызов нужной функции
    f = methods_map.get(method)
    if f is None:
        # на всякий случай нормализуем ключи
        key = str(method).strip().upper()
        if key == "HW" or "HOLT" in key:
            f = fc_hw_like_training
        elif "SARIMA" in key or "ARIMA" in key:
            f = fc_sarima_like_training
        else:
            f = fc_prophet_like_training

    fc = f(train)
    actual = fact_aug.get(region, np.nan)
    pct_dev = (fc - actual) / actual * 100 if pd.notna(actual) else np.nan

    rows.append({
        "Регион": region,
        "Лучший метод": method,
        "Прогноз (2025-08)": round(fc, 2),
        "Факт (2025-08)": round(actual, 2) if pd.notna(actual) else np.nan,
        "Отклонение, %": round(pct_dev, 2) if pd.notna(pct_dev) else np.nan
    })

results_aug = pd.DataFrame(rows).sort_values("Регион").reset_index(drop=True)
results_aug.to_excel("results/Яйца - Прогноз на 2025-08 (как в обучении).xlsx")
results_aug

,Регион,Лучший метод,Прогноз (2025-08),Факт (2025-08),"Отклонение, %"
0,АКМОЛИНСКАЯ ОБЛАСТЬ,HW,59152.27,58844.8,0.52
1,АКТЮБИНСКАЯ ОБЛАСТЬ,HW,19140.53,20835.1,-8.13
2,АЛМАТИНСКАЯ ОБЛАСТЬ,SARIMA,36618.34,42994.8,-14.83
3,АТЫРАУСКАЯ ОБЛАСТЬ,HW,3845.81,4209.1,-8.63
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,SARIMA,5545.93,5929.0,-6.46
5,ГАЛМАТЫ,SARIMA,14.59,NaN,NaN
6,ГАСТАНА,SARIMA,0.14,NaN,NaN
7,ГШЫМКЕНТ,HW,22508.85,NaN,NaN
8,ЖАМБЫЛСКАЯ ОБЛАСТЬ,HW,13448.47,14890.0,-9.68
9,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,SARIMA,11102.95,12202.0,-9.01
